In [49]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import (
    KFold, StratifiedKFold, RepeatedKFold, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn import datasets
from sklearn.metrics import accuracy_score as tools
from sklearn.metrics import classification_report
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

In [50]:
!pip install xgboost catboost

In [51]:
df= pd.read_csv('proje.csv') #df tanımladık ve mevcut veri setimizi ekledik .
df.info()  #veri setimiz hakkında bilgilendirme

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2149 entries, 0 to 2148
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PatientID                  2149 non-null   int64  
 1   Age                        2149 non-null   int64  
 2   Gender                     2149 non-null   int64  
 3   Ethnicity                  2149 non-null   int64  
 4   EducationLevel             2149 non-null   int64  
 5   BMI                        2149 non-null   float64
 6   Smoking                    2149 non-null   int64  
 7   AlcoholConsumption         2140 non-null   float64
 8   PhysicalActivity           2140 non-null   float64
 9   DietQuality                2139 non-null   float64
 10  SleepQuality               2149 non-null   float64
 11  FamilyHistoryAlzheimers    2149 non-null   int64  
 12  CardiovascularDisease      2149 non-null   int64  
 13  Diabetes                   2149 non-null   int64

In [52]:
#eksik verilerin bulunuduğu sütuna göre ortalamasının alınması

In [53]:
df["AlcoholConsumption"].fillna(df["AlcoholConsumption"].mean(), inplace=True) 
df["PhysicalActivity"].fillna(df["PhysicalActivity"].mean(), inplace=True)
df["DietQuality"].fillna(df["DietQuality"].mean(), inplace=True)
#eksik değerler bulunun sütunların ortalamasını alıp o sutündaki boşluklara yerleştirdik. 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2149 entries, 0 to 2148
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PatientID                  2149 non-null   int64  
 1   Age                        2149 non-null   int64  
 2   Gender                     2149 non-null   int64  
 3   Ethnicity                  2149 non-null   int64  
 4   EducationLevel             2149 non-null   int64  
 5   BMI                        2149 non-null   float64
 6   Smoking                    2149 non-null   int64  
 7   AlcoholConsumption         2149 non-null   float64
 8   PhysicalActivity           2149 non-null   float64
 9   DietQuality                2149 non-null   float64
 10  SleepQuality               2149 non-null   float64
 11  FamilyHistoryAlzheimers    2149 non-null   int64  
 12  CardiovascularDisease      2149 non-null   int64  
 13  Diabetes                   2149 non-null   int64

C:\Users\kemal\AppData\Local\Temp\ipykernel_8872\2057996597.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["AlcoholConsumption"].fillna(df["AlcoholConsumption"].mean(), inplace=True)
C:\Users\kemal\AppData\Local\Temp\ipykernel_8872\2057996597.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

In [54]:
#aykrırı değerlerin tespiti ve işaretlenmesi

In [55]:
#aykırı değer kontrolü yapılacak sayısal sutünları bir dizi de listeledik.
aykirilik_kontrolu_gereken_sutunlar = [
    "Age",
    "BMI",
    "AlcoholConsumption",
    "PhysicalActivity",
    "DietQuality",
    "SleepQuality",
    "SystolicBP",
    "DiastolicBP",
    "CholesterolTotal",
    "CholesterolLDL",
    "CholesterolHDL",
    "CholesterolTriglycerides",
    "MMSE",
    "FunctionalAssessment",
    "ADL"
]

In [56]:
for sutun in aykirilik_kontrolu_gereken_sutunlar: #sutun adlı bir değişken tanımladık ve oluşturduğumuz dizi içerisinde döndürüyoruz.
    Q1 = df[sutun].quantile(0.25)  #tüm sütunlarda kendi içerisindeki verilerin 1. çeyrek (Q1) değerini bulur. Yani alt %25’lik sınır.
    Q3 = df[sutun].quantile(0.75)  #aynı şekilde 3. çeyrek (Q3) değeri. Üst %25’lik sınır.
    IQR = Q3 - Q1  #Verinin yayılımını ölçer, orta %50'lik kısmın uzunluğudur.
 
    alt_sinir = Q1 - 1.5 * IQR
    ust_sinir = Q3 + 1.5 * IQR
#aykırı değerler için alt ve üst değerleri hesapladık
    
    aykiri_veriler = df[(df[sutun] <= alt_sinir) & (df[sutun] >= ust_sinir)]
    print(f"{sutun}: {len(aykiri_veriler)} aykırı değer")
    
#aykiri_degerler değişkeni oluşturduk, her sütun için alt ve üst sınırlar dışında kalan değerleri atadık, print ile bu sütunların adını ve değerlerin 
#uzunluğunu ekrana yazdırdık

Age: 0 aykırı değer
BMI: 0 aykırı değer
AlcoholConsumption: 0 aykırı değer
PhysicalActivity: 0 aykırı değer
DietQuality: 0 aykırı değer
SleepQuality: 0 aykırı değer
SystolicBP: 0 aykırı değer
DiastolicBP: 0 aykırı değer
CholesterolTotal: 0 aykırı değer
CholesterolLDL: 0 aykırı değer
CholesterolHDL: 0 aykırı değer
CholesterolTriglycerides: 0 aykırı değer
MMSE: 0 aykırı değer
FunctionalAssessment: 0 aykırı değer
ADL: 0 aykırı değer


In [57]:
(df[sutun] >= alt_sinir) & (df[sutun] <= ust_sinir)

0       True
1       True
2       True
3       True
4       True
        ... 
2144    True
2145    True
2146    True
2147    True
2148    True
Name: ADL, Length: 2149, dtype: bool

In [58]:
df = df.drop(columns=['PatientID'])
df

,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,9.025679,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,7.151293,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,9.673574,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,8.392554,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,5.597238,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144,61,0,0,1,39.121757,0,1.561126,4.049964,6.555306,7.535540,...,0,0,4.492838,1,0,0,0,0,1,XXXConfid
2145,75,0,0,2,17.857903,0,18.767261,1.360667,2.904662,8.555256,...,0,1,9.204952,0,0,0,0,0,1,XXXConfid
2146,77,0,0,1,15.476479,0,4.594670,9.886002,8.120025,5.769464,...,0,0,5.036334,0,0,0,0,0,1,XXXConfid
2147,78,1,3,1,15.299911,0,8.674505,6.354282,1.263427,8.322874,...,0,0,3.785399,0,0,0,0,1,1,XXXConfid


In [59]:
df = df.drop(columns=['DoctorInCharge'])
df

,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,...,FunctionalAssessment,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis
0,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,9.025679,...,6.518877,0,0,1.725883,0,0,0,1,0,0
1,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,7.151293,...,7.118696,0,0,2.592424,0,0,0,0,1,0
2,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,9.673574,...,5.895077,0,0,7.119548,0,1,0,1,0,0
3,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,8.392554,...,8.965106,0,1,6.481226,0,0,0,0,0,0
4,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,5.597238,...,6.045039,0,0,0.014691,0,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144,61,0,0,1,39.121757,0,1.561126,4.049964,6.555306,7.535540,...,0.238667,0,0,4.492838,1,0,0,0,0,1
2145,75,0,0,2,17.857903,0,18.767261,1.360667,2.904662,8.555256,...,8.687480,0,1,9.204952,0,0,0,0,0,1
2146,77,0,0,1,15.476479,0,4.594670,9.886002,8.120025,5.769464,...,1.972137,0,0,5.036334,0,0,0,0,0,1
2147,78,1,3,1,15.299911,0,8.674505,6.354282,1.263427,8.322874,...,5.173891,0,0,3.785399,0,0,0,0,1,1


In [60]:
#normalizasyon

In [61]:
normalize_edilecek_sutunlar = [
    "Age",
    "BMI",
    "AlcoholConsumption",
    "PhysicalActivity",
    "DietQuality",
    "SleepQuality",
    "SystolicBP",
    "DiastolicBP",
    "CholesterolTotal",
    "CholesterolLDL",
    "CholesterolHDL",
    "CholesterolTriglycerides",
    "MMSE",
    "FunctionalAssessment",
    "ADL"
]


In [62]:
scaler = RobustScaler()

# Veriyi dönüştür
scaled_array = scaler.fit_transform(df)

# Yeni dataframe'e aktar
df_scaled = pd.DataFrame(scaled_array, columns=df.columns)

print(df_scaled)

         Age  Gender  Ethnicity  EducationLevel       BMI  Smoking  \
0    -0.1250    -1.0        0.0             1.0 -0.399415      0.0   
1     0.8750    -1.0        0.0            -1.0 -0.081270      0.0   
2    -0.1250    -1.0        3.0             0.0 -0.818057      0.0   
3    -0.0625     0.0        0.0             0.0  0.487577      1.0   
4     0.8750    -1.0        0.0            -1.0 -0.579763      0.0   
...      ...     ...        ...             ...       ...      ...   
2144 -0.8750    -1.0        0.0             0.0  0.921642      0.0   
2145  0.0000    -1.0        0.0             1.0 -0.812997      0.0   
2146  0.1250    -1.0        0.0             0.0 -1.007266      0.0   
2147  0.1875     0.0        3.0             0.0 -1.021670      0.0   
2148 -0.1875    -1.0        0.0             1.0  0.445884      0.0   

      AlcoholConsumption  PhysicalActivity  DietQuality  SleepQuality  ...  \
0               0.336159          0.320341    -0.734969      0.620236  ...   
1  

In [63]:
df_scaled

,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,...,FunctionalAssessment,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis
0,-0.1250,-1.0,0.0,1.0,-0.399415,0.0,0.336159,0.320341,-0.734969,0.620236,...,0.285992,0.0,0.0,-0.632431,0.0,0.0,0.0,1.0,0.0,0.0
1,0.8750,-1.0,0.0,-1.0,-0.081270,0.0,-0.542783,0.588259,-0.899138,0.011575,...,0.406420,0.0,0.0,-0.467019,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.1250,-1.0,3.0,0.0,-0.818057,0.0,0.964428,0.634910,-0.640025,0.830625,...,0.160748,0.0,0.0,0.397158,0.0,1.0,0.0,1.0,0.0,0.0
3,-0.0625,0.0,0.0,0.0,0.487577,1.0,0.226932,0.755735,0.471532,0.414644,...,0.777133,0.0,1.0,0.275310,0.0,0.0,0.0,0.0,0.0,0.0
4,0.8750,-1.0,0.0,-1.0,-0.579763,0.0,0.853918,0.316890,-0.844300,-0.493066,...,0.190857,0.0,0.0,-0.959079,0.0,0.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144,-0.8750,-1.0,0.0,0.0,0.921642,0.0,-0.842106,-0.151581,0.297088,0.136350,...,-0.974917,0.0,0.0,-0.104251,1.0,0.0,0.0,0.0,0.0,1.0
2145,0.0000,-1.0,0.0,1.0,-0.812997,0.0,0.885333,-0.708918,-0.426339,0.467478,...,0.721393,0.0,1.0,0.795239,0.0,0.0,0.0,0.0,0.0,1.0
2146,0.1250,-1.0,0.0,0.0,-1.007266,0.0,-0.537548,1.057895,0.607159,-0.437140,...,-0.626880,0.0,0.0,-0.000504,0.0,0.0,0.0,0.0,0.0,1.0
2147,0.1875,0.0,3.0,0.0,-1.021670,0.0,-0.127946,0.325972,-0.751573,0.392018,...,0.015952,0.0,0.0,-0.239293,0.0,0.0,0.0,0.0,1.0,1.0


In [ ]:
#cross validation

In [64]:
y = df['Diagnosis']
y = LabelEncoder().fit_transform(y)

In [65]:
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(df.drop('Diagnosis', axis=1)))
X.columns = df.drop('Diagnosis', axis=1).columns

In [94]:
# Model
model = RandomForestClassifier(random_state=42)

# Cross-validation stratejileri
strategies = {
    "KFold": KFold(n_splits=8, shuffle=True, random_state=42),
    "StratifiedKFold": StratifiedKFold(n_splits=8, shuffle=True, random_state=42),
    "RepeatedKFold": RepeatedKFold(n_splits=8, n_repeats=3, random_state=42),
}

# Sonuçları tut
results = {}

# Skorları hesapla
for name, strategy in strategies.items():
    scores = cross_val_score(model, X, y, cv=strategy, scoring='accuracy')
    results[name] = scores
    print(f"{name} - Ortalama Başarı: {np.mean(scores): .4f} | Skorlar: {scores}")


KFold - Ortalama Başarı:  0.9404 | Skorlar: [0.94423792 0.92565056 0.92193309 0.96654275 0.95910781 0.95895522
 0.92537313 0.92164179]
StratifiedKFold - Ortalama Başarı:  0.9386 | Skorlar: [0.94795539 0.94052045 0.91078067 0.94052045 0.94423792 0.95895522
 0.92910448 0.93656716]
RepeatedKFold - Ortalama Başarı:  0.9401 | Skorlar: [0.94423792 0.92565056 0.92193309 0.96654275 0.95910781 0.95895522
 0.92537313 0.92164179 0.94052045 0.95910781 0.9330855  0.92193309
 0.97026022 0.93656716 0.91791045 0.94402985 0.93680297 0.94052045
 0.93680297 0.94052045 0.94423792 0.92537313 0.94029851 0.95149254]


In [91]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report
# Özellikler ve hedef değişken
X = df.drop(columns=["Diagnosis"])
y = df["Diagnosis"]

# Veriyi eğitim ve test setlerine ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Modelleri tanımla
models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Support Vector Machine": SVC(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# Tüm modelleri eğit, tahmin yap ve sonuçları yazdır
for model_name, model in models.items():
    print(f"------ {model_name} ------")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))

------ Decision Tree ------
Accuracy: 0.8728682170542635
Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.91      0.90       401
           1       0.85      0.81      0.83       244

    accuracy                           0.87       645
   macro avg       0.87      0.86      0.86       645
weighted avg       0.87      0.87      0.87       645

------ Random Forest ------
Accuracy: 0.8976744186046511
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.97      0.92       401
           1       0.94      0.77      0.85       244

    accuracy                           0.90       645
   macro avg       0.91      0.87      0.89       645
weighted avg       0.90      0.90      0.90       645

------ K-Nearest Neighbors ------
Accuracy: 0.5488372093023256
Classification Report:
               precision    recall  f1-score   support

           0       0.62      0.73      0.6

C:\YAZILIM_GELISTIRME\Anaconda\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.8108527131782945
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.88      0.85       401
           1       0.77      0.70      0.74       244

    accuracy                           0.81       645
   macro avg       0.80      0.79      0.80       645
weighted avg       0.81      0.81      0.81       645

------ Support Vector Machine ------
Accuracy: 0.6217054263565891
Classification Report:
               precision    recall  f1-score   support

           0       0.62      1.00      0.77       401
           1       0.00      0.00      0.00       244

    accuracy                           0.62       645
   macro avg       0.31      0.50      0.38       645
weighted avg       0.39      0.62      0.48       645

------ XGBoost ------


C:\YAZILIM_GELISTIRME\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\YAZILIM_GELISTIRME\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\YAZILIM_GELISTIRME\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\YAZILIM_GELISTIRME\

Accuracy: 0.9410852713178295
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.97      0.95       401
           1       0.95      0.89      0.92       244

    accuracy                           0.94       645
   macro avg       0.94      0.93      0.94       645
weighted avg       0.94      0.94      0.94       645

------ CatBoost ------
Accuracy: 0.9410852713178295
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.97      0.95       401
           1       0.95      0.89      0.92       244

    accuracy                           0.94       645
   macro avg       0.94      0.93      0.94       645
weighted avg       0.94      0.94      0.94       645

